# Build the mechanics dataset

Run this notebook first. It merges and deduplicates the two mechanics lists, saves `Data/boardgames_prepared.csv`, and caches name embeddings in `Data/mechanic_embeddings.npz` for semantic grouping.

In [1]:
from ast import literal_eval
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

folder = Path("Boardgames") if Path("Boardgames").is_dir() else Path(".")
data_dir = folder / "Data"
raw = pd.read_csv(data_dir / "combined_boardgames.csv")
assert raw["BGGId"].notna().all() and raw["BGGId"].is_unique

## Merge and deduplicate mechanics

Take the union of both source lists for each game. Match formatting variants using case, whitespace, commas and slash spacing, and prefer the detailed source's spelling. Distinct names such as `Card Drafting` and `Open Drafting` stay separate. Missing in both sources stays missing.

Normalize the verified source alias `Time Track` to `Turn Order: Time Track` before combining the lists. Both describe turn order determined by position on a time track. [Current BGG mechanic](https://boardgamegeek.com/boardgamemechanic/2663/turn-order-time-track) · [Earlier BGG terminology](https://boardgamegeek.com/wiki/page/Mechanics)

In [2]:
MECHANIC_ALIASES = {"time track": "Turn Order: Time Track"}


def name_key(name):
    text = unicodedata.normalize("NFKC", name).casefold().replace(",", "")
    return re.sub(r"\s*/\s*", "/", " ".join(text.split()))


def parse_list(value):
    if pd.isna(value) or not value.strip():
        return None
    return [MECHANIC_ALIASES.get(name_key(name), name.strip()) for name in literal_eval(value)]


original = raw["Mechanics"].map(parse_list)
detailed = raw["boardgamemechanic"].map(parse_list)
names = {}
for labels in pd.concat([detailed, original]):
    for name in labels or []:
        names.setdefault(name_key(name), name.strip())

combined = pd.Series([
    None if a is None and b is None else sorted({names[name_key(n)] for n in (a or []) + (b or [])})
    for a, b in zip(original, detailed)
], index=raw.index, name="MechanicsCombined")

## Select fields and one-hot encode

Keep IDs and names as labels, `AvgRating` as the outcome, and the seven fields below as predictors. Treat zero as missing in year, player limits, playtime and age. Drop predictors with over 20% missing. Each mechanic gets a 0/1 column. Missing combined lists produce missing indicators.

In [3]:
features = ["YearPublished", "MinPlayers", "MaxPlayers", "MfgPlaytime", "MfgAgeRec",
            "IsReimplementation", "Kickstarted"]
labels = ["BGGId", "Name", "AvgRating"]
dataset = raw[labels + features].copy()
for column in ["AvgRating"] + features:
    dataset[column] = pd.to_numeric(dataset[column], errors="coerce")
zero_missing = ["YearPublished", "MinPlayers", "MaxPlayers", "MfgPlaytime", "MfgAgeRec"]
dataset[zero_missing] = dataset[zero_missing].replace(0, np.nan)
for column in ["IsReimplementation", "Kickstarted"]:
    dataset[column] = dataset[column].where(dataset[column].isin([0, 1]))
missing = dataset[features].isna().mean()
dataset = dataset[labels + list(missing[missing <= 0.20].index)]

encoder = MultiLabelBinarizer()
indicators = pd.DataFrame(
    encoder.fit_transform(combined.map(lambda labels: labels or [])),
    columns=[f"Mechanic: {name}" for name in encoder.classes_],
    index=raw.index, dtype="UInt8",
)
indicators.loc[combined.isna(), :] = pd.NA
dataset = pd.concat([dataset, combined, indicators], axis=1)

In [4]:
output = data_dir / "boardgames_prepared.csv"
dataset.to_csv(output, index=False)
print(f"Saved {len(dataset):,} games and {len(indicators.columns)} mechanic indicators to {output}.")
print(f"{combined.isna().sum():,} games have no mechanics list.")
display(dataset[["Name", "MechanicsCombined"]].head(3))

Saved 21,925 games and 195 mechanic indicators to Data/boardgames_prepared.csv.
1,732 games have no mechanics list.


,Name,MechanicsCombined
0,Die Macher,"[Alliances, Area Majority / Influence, Auction..."
1,Dragonmaster,[Trick-taking]
2,Samurai,"[Area Majority / Influence, Hand Management, H..."


## Embed mechanic names locally

Use the dedicated `all-minilm` model through local Ollama. The first rebuild requires Ollama running and `ollama pull all-minilm`. Vectors are cached with mechanic names and model identity. Set `REBUILD_EMBEDDINGS = True` to refresh them. The analysis notebook reads the cache and can run without a model server.

In [5]:
import json
import urllib.request

EMBEDDING_MODEL = "all-minilm:latest"
REBUILD_EMBEDDINGS = False
embedding_cache = data_dir / "mechanic_embeddings.npz"
mechanic_names = encoder.classes_.astype(str)
cached = False
if embedding_cache.exists() and not REBUILD_EMBEDDINGS:
    with np.load(embedding_cache, allow_pickle=False) as saved:
        cached = (saved["model"].item() == EMBEDDING_MODEL
                  and np.array_equal(saved["names"], mechanic_names))
if not cached:
    vectors = []
    for start in range(0, len(mechanic_names), 32):
        request = urllib.request.Request(
            "http://127.0.0.1:11434/api/embed",
            data=json.dumps({"model": EMBEDDING_MODEL,
                             "input": mechanic_names[start:start + 32].tolist(),
                             "keep_alive": "1m"}).encode(),
            headers={"Content-Type": "application/json"},
        )
        with urllib.request.urlopen(request, timeout=120) as response:
            vectors.extend(json.load(response)["embeddings"])
    vectors = np.asarray(vectors, dtype=np.float32)
    if vectors.shape[0] != len(mechanic_names) or not np.isfinite(vectors).all():
        raise ValueError("Invalid mechanic embeddings.")
    with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=10) as response:
        installed = json.load(response)["models"]
    digest = next(m["digest"] for m in installed if m["name"] == EMBEDDING_MODEL)
    np.savez_compressed(embedding_cache, names=mechanic_names, vectors=vectors,
                        model=EMBEDDING_MODEL, model_digest=digest)
with np.load(embedding_cache, allow_pickle=False) as saved:
    print(f"Cached {len(saved['names'])} mechanic names with {saved['vectors'].shape[1]}-dimensional embeddings.")

Cached 195 mechanic names with 384-dimensional embeddings.
